In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("Number of GPUs:", torch.cuda.device_count())
    
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
else:
    print("No GPU detected!")

In [ ]:
import json
from pathlib import Path
from collections import Counter

# Find uploaded dataset automatically
matches = list(Path("/kaggle/input").rglob("corpus.jsonl"))

if not matches:
    raise FileNotFoundError(
        "corpus.jsonl not found! Please upload the dataset."
    )

DATA_DIR = matches[0].parent
print("Dataset directory:", DATA_DIR)

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

corpus = load_jsonl(DATA_DIR / "corpus.jsonl")
questions = load_jsonl(DATA_DIR / "questions_dev.jsonl")
labels = load_jsonl(DATA_DIR / "labels_dev.jsonl")

print("\nDataset information:")
print("Documents:", len(corpus))
print("Questions:", len(questions))
print("Labels:", len(labels))

# Validate dataset
doc_ids = {d["doc_id"] for d in corpus}
qids = {q["qid"] for q in questions}
label_qids = {x["qid"] for x in labels}

assert len(doc_ids) == len(corpus)
assert qids == label_qids
assert all(
    set(label["gold_doc_ids"]).issubset(doc_ids)
    for label in labels
)

print("\nQuestion types:")
for qtype, count in Counter(q["type"] for q in questions).items():
    print(f"{qtype}: {count}")

print("\nDataset validation: PASSED")


In [ ]:
%pip install -q -U "sentence-transformers>=5.0,<6" "transformers>=4.51,<5"

In [ ]:

%pip install -q -U "sentence-transformers>=6.1.0" accelerate pillow
%pip install -q -U "git+https://github.com/huggingface/transformers.git"


In [ ]:

import transformers
import sentence_transformers

from transformers.models.embedding_gemma2 import EmbeddingGemma2Config

print("Transformers:", transformers.__version__)
print("Sentence Transformers:", sentence_transformers.__version__)
print("EmbeddingGemma 2 architecture: SUPPORTED")


In [ ]:
import gc
import torch

del model
gc.collect()
torch.cuda.empty_cache()

print("GPU cache cleared.")

In [ ]:

# import gc
# import time
# import torch
# import numpy as np
# from pathlib import Path
# from sentence_transformers import SentenceTransformer

# # Free previous model
# if "model" in globals():
#     del model

# gc.collect()
# torch.cuda.empty_cache()

# MODEL_ID = "BAAI/bge-small-en-v1.5"

# SAVE_DIR = Path(
#     "/kaggle/working/embedding_experiments/bge_small_en_v1_5"
# )
# SAVE_DIR.mkdir(parents=True, exist_ok=True)

# start_time = time.perf_counter()

# model = SentenceTransformer(
#     MODEL_ID,
#     device="cuda:0"
# )

# # BGE supports up to 512 tokens
# model.max_seq_length = 512

# # Same document format used in Qwen experiments
# passage_texts = [
#     f"{doc['title']}\n{doc['text']}"
#     for doc in corpus
# ]

# # BGE recommended retrieval instruction
# INSTRUCTION = (
#     "Represent this sentence for searching relevant passages: "
# )

# question_texts = [
#     INSTRUCTION + q["question"]
#     for q in questions
# ]

# print("Encoding documents...")

# doc_embeddings = model.encode(
#     passage_texts,
#     prompt="",
#     batch_size=32,
#     normalize_embeddings=True,
#     convert_to_numpy=True,
#     show_progress_bar=True
# )

# print("Encoding questions...")

# query_embeddings = model.encode(
#     question_texts,
#     prompt="",
#     batch_size=32,
#     normalize_embeddings=True,
#     convert_to_numpy=True,
#     show_progress_bar=True
# )

# assert doc_embeddings.shape == (3882, 384)
# assert query_embeddings.shape == (300, 384)

# assert np.isfinite(doc_embeddings).all()
# assert np.isfinite(query_embeddings).all()

# np.save(SAVE_DIR / "doc_embeddings.npy", doc_embeddings)
# np.save(SAVE_DIR / "query_embeddings.npy", query_embeddings)

# print("Document embeddings:", doc_embeddings.shape)
# print("Query embeddings:", query_embeddings.shape)
# print("Elapsed seconds:", round(time.perf_counter() - start_time, 1))
# print("Saved successfully!")


In [ ]:

# import time
# import numpy as np
# import torch
# from pathlib import Path
# from sentence_transformers import SentenceTransformer

# MODEL_ID = "google/embeddinggemma-2"

# SAVE_DIR = Path(
#     "/kaggle/working/embedding_experiments/embeddinggemma_2"
# )
# SAVE_DIR.mkdir(parents=True, exist_ok=True)

# start_time = time.perf_counter()

# model = SentenceTransformer(
#     MODEL_ID,
#     device="cuda:0",
#     model_kwargs={"torch_dtype": torch.float32},
#     config_kwargs={
#         "vision_config": None,
#         "audio_config": None
#     }
# )

# model.max_seq_length = 2048

# print("Model:", MODEL_ID)
# print("Dimension:", model.get_sentence_embedding_dimension())
# print("QuestionAnswering prompt:",
#       model.prompts.get("QuestionAnswering"))

# # Document format recommended by Google
# passage_texts = [
#     f"title: {doc['title']} | text: {doc['text']}"
#     for doc in corpus
# ]

# question_texts = [
#     q["question"] for q in questions
# ]

# # Encode documents; prefix is already included manually
# doc_embeddings = model.encode(
#     passage_texts,
#     prompt="",
#     batch_size=4,
#     show_progress_bar=True,
#     convert_to_numpy=True,
#     normalize_embeddings=True
# )

# # Encode queries with the QA-specific prompt
# query_embeddings = model.encode(
#     question_texts,
#     prompt_name="QuestionAnswering",
#     batch_size=4,
#     show_progress_bar=True,
#     convert_to_numpy=True,
#     normalize_embeddings=True
# )

# assert doc_embeddings.shape == (3882, 768)
# assert query_embeddings.shape == (300, 768)

# assert np.isfinite(doc_embeddings).all()
# assert np.isfinite(query_embeddings).all()

# np.save(SAVE_DIR / "doc_embeddings.npy", doc_embeddings)
# np.save(SAVE_DIR / "query_embeddings.npy", query_embeddings)

# print("Document embeddings:", doc_embeddings.shape)
# print("Query embeddings:", query_embeddings.shape)
# print("Elapsed seconds:", round(time.perf_counter() - start_time, 1))
# print("Saved successfully!")


In [ ]:
# A1: shared run configuration; this cell does NOT load any model.
from pathlib import Path
import os
import sys


def locate_a1_project():
    options = []
    if os.environ.get("A1_PROJECT_ROOT"):
        options.append(Path(os.environ["A1_PROJECT_ROOT"]))
    options.extend([Path.cwd(), *Path.cwd().parents])
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.is_dir():
        # Add the private project source/configs as a Kaggle Input dataset for replay.
        options.extend(p.parent.parent for p in kaggle_input.rglob("configs/dense.yaml"))
    for root in options:
        if (root / "src" / "config.py").is_file() and all(
            (root / "configs" / f"{name}.yaml").is_file()
            for name in ("closed_book", "oracle", "dense")
        ):
            return root.resolve()
    raise FileNotFoundError(
        "A1 project files not found. Set A1_PROJECT_ROOT or attach "
        "the repository files as a Kaggle Input dataset."
    )

A1_ROOT = locate_a1_project()
if str(A1_ROOT) not in sys.path:
    sys.path.insert(0, str(A1_ROOT))

from src.config import load_config
CLOSED_CFG = load_config(A1_ROOT / "configs/closed_book.yaml")
ORACLE_CFG = load_config(A1_ROOT / "configs/oracle.yaml")
DENSE_CFG = load_config(A1_ROOT / "configs/dense.yaml")
DENSE_K = int(DENSE_CFG["settings"]["retrieval"]["k"])
A1_OUTPUT_ROOT = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else A1_ROOT


def a1_prediction_path(cfg, split):
    return A1_OUTPUT_ROOT / cfg["source_predictions"][split]


def a1_retrieval_path(split):
    return A1_OUTPUT_ROOT / DENSE_CFG["retrieval_predictions"][split]

assert CLOSED_CFG["models"]["llm"]["name"] == ORACLE_CFG["models"]["llm"]["name"]
assert ORACLE_CFG["models"]["llm"]["name"] == DENSE_CFG["models"]["llm"]["name"]


def a1_prompt(cfg):
    path = A1_ROOT / cfg["settings"]["prompt_file"]
    return path.read_text(encoding="utf-8").strip()


def a1_torch_dtype(torch, cfg):
    name = cfg["models"]["llm"]["compute_dtype"]
    if name not in {"float16", "bfloat16", "float32"}:
        raise ValueError(f"Unsupported torch dtype: {name}")
    return getattr(torch, name)

print("A1 loaded three YAML configurations from:", A1_ROOT)


In [ ]:
import torch
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer

# MODEL_ID = "Qwen/Qwen3-Embedding-0.6B"

# SAVE_DIR = Path("/kaggle/working/embedding_experiments/qwen3_0_6b")

EMBED_MODEL_ID = DENSE_CFG["models"]["embedder"]

SAVE_DIR = Path(
  "/kaggle/working/embedding_experiments/qwen3_4b"
)

SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Load embedding model
model = SentenceTransformer(
    EMBED_MODEL_ID,
    device="cuda",
    model_kwargs={"torch_dtype": getattr(torch, DENSE_CFG["settings"]["embedding"]["torch_dtype"])},
    tokenizer_kwargs={"padding_side": "left"}
)

# Context length: sufficient for our passages
model.max_seq_length = DENSE_CFG["settings"]["embedding"]["max_seq_length"]

print("Model loaded successfully!")
print("Embedding dimension:", model.get_sentence_embedding_dimension())
print("Query prompt:", model.prompts.get("query"))

# Prepare passage texts
passage_texts = [
    f"{doc['title']}\n{doc['text']}"
    for doc in corpus
]

# Prepare question texts
question_texts = [
    q["question"] for q in questions
]

assert "query" in model.prompts, "Query prompt is missing!"

# Encode corpus
print("\nEncoding documents...")

doc_embeddings = model.encode(
    passage_texts,
    batch_size=DENSE_CFG["settings"]["embedding"]["exploratory_batch_size"],
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=DENSE_CFG["settings"]["retrieval"]["normalize_embeddings"]
)

# Encode questions with Qwen query instruction
print("\nEncoding questions...")

query_embeddings = model.encode(
    question_texts,
    prompt_name="query",
    batch_size=DENSE_CFG["settings"]["embedding"]["exploratory_batch_size"],
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=DENSE_CFG["settings"]["retrieval"]["normalize_embeddings"]
)

# Save embeddings for reuse
np.save(SAVE_DIR / "doc_embeddings.npy", doc_embeddings)
np.save(SAVE_DIR / "query_embeddings.npy", query_embeddings)

print("\nDocument embeddings:", doc_embeddings.shape)
print("Question embeddings:", query_embeddings.shape)
print("Embeddings saved successfully!")


In [ ]:
# Convert to float32 for stable similarity calculations
D = doc_embeddings.astype(np.float32)
Q = query_embeddings.astype(np.float32)

# Exact cosine similarity: every query against every document
similarity_matrix = Q @ D.T

doc_id_list = [doc["doc_id"] for doc in corpus]
doc_id_to_index = {
    doc_id: i for i, doc_id in enumerate(doc_id_list)
}

def dense_search(query_index, k=None, exclude_ids=None):
    k = DENSE_K if k is None else int(k)
    scores = similarity_matrix[query_index].copy()

    if exclude_ids:
        for doc_id in exclude_ids:
            if doc_id in doc_id_to_index:
                scores[doc_id_to_index[doc_id]] = -np.inf

    # Exact ranking; masking happens BEFORE top-k
    ranked_indices = np.argsort(
        -scores, kind="stable"
    )

    result = []
    for idx in ranked_indices:
        if not np.isfinite(scores[idx]):
            continue
        result.append(doc_id_list[idx])
        if len(result) == k:
            break

    return result

# Retrieve top-10 for all development questions
retrieved_results = {}

for i, q in enumerate(questions):
    retrieved_results[q["qid"]] = dense_search(i, k=DENSE_K)

print("Similarity matrix shape:", similarity_matrix.shape)
print("Retrieval completed:", len(retrieved_results))

# Example
sample_q = questions[0]

print("\nQuestion:", sample_q["question"])
print("Top-10:", retrieved_results[sample_q["qid"]])


In [ ]:
import math
from collections import defaultdict

label_map = {x["qid"]: x for x in labels}

from src.metrics import retrieval_metrics

# Evaluate all questions
all_metrics = []
metrics_by_type = defaultdict(list)

for q in questions:
    qid = q["qid"]
    gold_ids = label_map[qid]["gold_doc_ids"]

    metrics = retrieval_metrics(
        retrieved_results[qid],
        gold_ids
    )

    all_metrics.append(metrics)
    metrics_by_type[q["type"]].append(metrics)

def print_metrics(rows, title):
    print(f"\n===== {title} (n={len(rows)}) =====")

    for metric in rows[0]:
        mean = np.mean([r[metric] for r in rows])
        print(f"{metric:12s}: {mean * 100:.2f}%")

print_metrics(all_metrics, "Overall")

for qtype, rows in sorted(metrics_by_type.items()):
    print_metrics(rows, qtype)


In [ ]:
import subprocess
import sys

pred_path = SAVE_DIR / "predictions.jsonl"

with open(pred_path, "w", encoding="utf-8") as f:
    for q in questions:
        row = {
            "qid": q["qid"],
            "answer": "",
            "retrieved": retrieved_results[q["qid"]]
        }
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

# Locate course scoring script
score_files = list(Path("/kaggle/input").rglob("score.py"))

if not score_files:
    raise FileNotFoundError("Please upload score.py to Kaggle.")

score_path = score_files[0]

cmd = [
    sys.executable,
    str(score_path),
    str(pred_path),
    "--pack", str(DATA_DIR),
    "--split", "dev"
]

subprocess.run(cmd, check=True)

In [ ]:

import subprocess
import sys
from pathlib import Path

baseline = Path(
    "/kaggle/working/embedding_experiments/qwen3_0_6b/predictions.jsonl"
)

candidate = Path(
    "/kaggle/working/embedding_experiments/qwen3_4b/predictions.jsonl"
)

assert baseline.exists(), "Baseline predictions not found!"
assert candidate.exists(), "4B predictions not found!"

result = subprocess.run(
    [
        sys.executable,
        str(score_path),
        str(candidate),
        "--pack", str(DATA_DIR),
        "--split", "dev",
        "--compare", str(baseline)
    ],
    capture_output=True,
    text=True,
    check=True
)

# Display only the paired comparison section
print("PAIRED COMPARISON")
print(result.stdout.split("paired difference:")[-1])


# LLM Selection

In [ ]:
%pip install -q -U bitsandbytes accelerate

In [ ]:

import json
import torch
import transformers
import bitsandbytes as bnb

from pathlib import Path
from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,
    BitsAndBytesConfig
)

print("Transformers:", transformers.__version__)
print("PyTorch:", torch.__version__)
print("BitsAndBytes:", bnb.__version__)

DATA_DIR = next(
    Path("/kaggle/input").rglob("labels_dev.jsonl")
).parent

def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [
            json.loads(line)
            for line in f
            if line.strip()
        ]

corpus = load_jsonl(DATA_DIR / "corpus.jsonl")
questions = load_jsonl(DATA_DIR / "questions_dev.jsonl")
labels = load_jsonl(DATA_DIR / "labels_dev.jsonl")

doc_map = {d["doc_id"]: d for d in corpus}
label_map = {x["qid"]: x for x in labels}

print("Documents:", len(corpus))
print("Questions:", len(questions))
print("Labels:", len(labels))
print("Dataset path:", DATA_DIR)


In [ ]:
import time
from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,
    BitsAndBytesConfig
)

MODEL_ID = ORACLE_CFG["models"]["llm"]["name"]

quant_config = BitsAndBytesConfig(
    load_in_4bit=ORACLE_CFG["models"]["llm"]["load_in_4bit"],
    bnb_4bit_quant_type=ORACLE_CFG["models"]["llm"]["quantization"],
    bnb_4bit_use_double_quant=ORACLE_CFG["models"]["llm"]["double_quantization"],
    bnb_4bit_compute_dtype=a1_torch_dtype(torch, ORACLE_CFG)
)

start = time.perf_counter()

print("Loading processor...")
processor = AutoProcessor.from_pretrained(MODEL_ID)

print("Loading Qwen3.5-9B in 4-bit...")

llm = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    dtype=getattr(torch, ORACLE_CFG["models"]["llm"]["model_dtype"]),
    device_map=ORACLE_CFG["models"]["llm"]["device_map"],
    attn_implementation=ORACLE_CFG["models"]["llm"]["attention_implementation"],
    low_cpu_mem_usage=ORACLE_CFG["models"]["llm"]["low_cpu_mem_usage"]
)

llm.eval()

print("\nModel loaded successfully!")
print("Loading time:", round(time.perf_counter() - start, 1), "seconds")
print("Device map:", llm.hf_device_map)

for i in range(torch.cuda.device_count()):
    allocated = torch.cuda.memory_allocated(i) / 1024**3
    print(f"GPU {i} allocated: {allocated:.2f} GB")


In [ ]:
SYSTEM_PROMPT = a1_prompt(ORACLE_CFG)


def build_oracle_messages(question):
    qid = question["qid"]
    gold_ids = label_map[qid]["gold_doc_ids"]

    contexts = []

    for i, doc_id in enumerate(gold_ids, start=1):
        doc = doc_map[doc_id]

        context = (
            f"[Passage {i} | {doc_id}]\n"
            f"Title: {doc['title']}\n"
            f"Text: {doc['text']}"
        )

        contexts.append(context)

    user_prompt = (
        "Passages:\n\n"
        + "\n\n".join(contexts)
        + "\n\nQuestion: "
        + question["question"]
        + "\n\nFinal answer:"
    )

    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt}
    ]


# Inspect one Oracle prompt
sample = questions[0]
sample_messages = build_oracle_messages(sample)

print("Question:", sample["question"])
print("Gold answer:", label_map[sample["qid"]]["answer"])
print("Gold documents:", label_map[sample["qid"]]["gold_doc_ids"])
print("\nPrompt preview:\n")
print(sample_messages[1]["content"][:1500])


In [ ]:

import re
import torch

@torch.inference_mode()
def generate_oracle_answer(question, max_new_tokens=None):
    if max_new_tokens is None:
        max_new_tokens = ORACLE_CFG["settings"]["generation"]["max_new_tokens"]

    messages = build_oracle_messages(question)

    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=ORACLE_CFG["settings"]["generation"]["enable_thinking"],
        return_dict=True,
        return_tensors="pt"
    )

    input_length = inputs["input_ids"].shape[1]

    print(f"Input tokens: {input_length}")

    if input_length > ORACLE_CFG["settings"]["generation"]["max_input_tokens"]:
        raise ValueError(
            f"Input context too long: {input_length} tokens"
        )

    inputs = inputs.to(ORACLE_CFG["settings"]["generation"]["device"])

    outputs = llm.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=ORACLE_CFG["settings"]["generation"]["do_sample"],
        pad_token_id=processor.tokenizer.eos_token_id
    )

    generated_ids = outputs[0][input_length:]

    raw_answer = processor.tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    # Remove thinking tags if unexpectedly present
    answer = re.sub(
        r"<think>.*?</think>",
        "",
        raw_answer,
        flags=re.DOTALL
    ).strip()

    # Remove common answer prefixes
    answer = re.sub(
        r"^(final answer|answer)\s*:\s*",
        "",
        answer,
        flags=re.IGNORECASE
    ).strip()

    answer = answer.split("\n")[0].strip().strip('"')

    return answer


In [ ]:
import json
import random
from pathlib import Path
from collections import Counter

EXPERIMENT_DIR = Path(
    "/kaggle/working/llm_experiments/oracle_50"
)
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_PATH = EXPERIMENT_DIR / "sample_qids.json"

SAMPLE_COUNTS = {
    "comparison": 12,
    "bridge_comparison": 11,
    "compositional": 21,
    "inference": 6
}

# Reuse the same sample if it already exists
if SAMPLE_PATH.exists():
    selected_qids = json.loads(
        SAMPLE_PATH.read_text(encoding="utf-8")
    )
else:
    rng = random.Random(42)
    selected_qids = []

    for qtype, n in SAMPLE_COUNTS.items():
        candidates = sorted(
            q["qid"]
            for q in questions
            if q["type"] == qtype
        )

        selected_qids.extend(
            rng.sample(candidates, n)
        )

    SAMPLE_PATH.write_text(
        json.dumps(selected_qids, indent=2),
        encoding="utf-8"
    )

question_map = {q["qid"]: q for q in questions}

assert len(selected_qids) == 50
assert len(set(selected_qids)) == 50
assert all(qid in question_map for qid in selected_qids)

print("Selected questions:", len(selected_qids))
print("Distribution:")

print(Counter(
    question_map[qid]["type"]
    for qid in selected_qids
))

print("Sample saved at:", SAMPLE_PATH)


In [ ]:
import time
from pathlib import Path

QWEN_DIR = EXPERIMENT_DIR / "qwen3_5_9b"
QWEN_DIR.mkdir(parents=True, exist_ok=True)

PRED_PATH = QWEN_DIR / "predictions.jsonl"
TIME_PATH = QWEN_DIR / "timings.jsonl"

def read_jsonl_safe(path):
    if not path.exists():
        return []

    with open(path, encoding="utf-8") as f:
        return [
            json.loads(line)
            for line in f
            if line.strip()
        ]

# Load already completed questions
previous = read_jsonl_safe(PRED_PATH)

completed = {row["qid"] for row in previous}

assert len(previous) == len(completed), (
    "Duplicate qids found in predictions!"
)

assert completed.issubset(set(selected_qids))

print("Already completed:", len(completed))
print("Remaining:", 50 - len(completed))

for index, qid in enumerate(selected_qids, start=1):

    if qid in completed:
        continue

    q = question_map[qid]

    print(f"\n[{index}/50] {qid} | {q['type']}")

    start = time.perf_counter()

    # Same Oracle generation function as before
    answer = generate_oracle_answer(q)

    elapsed = time.perf_counter() - start

    prediction = {
        "qid": qid,
        "answer": answer,
        "retrieved": label_map[qid]["gold_doc_ids"]
    }

    timing = {
        "qid": qid,
        "elapsed_seconds": round(elapsed, 3)
    }

    # Save immediately after every question
    with open(PRED_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(
            prediction, ensure_ascii=False
        ) + "\n")

    with open(TIME_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(timing) + "\n")

    completed.add(qid)

    print("Predicted:", answer)
    print("Gold:", label_map[qid]["answer"])
    print("Time:", round(elapsed, 2), "seconds")
    print("Saved:", len(completed), "/ 50")

print("\nCompleted:", len(completed), "/ 50")
print("Predictions:", PRED_PATH)


In [ ]:
import subprocess
import sys

saved_predictions = read_jsonl_safe(PRED_PATH)

assert len(saved_predictions) == 50, (
    f"Expected 50 predictions, got {len(saved_predictions)}"
)

cmd = [
    sys.executable,
    str(score_path),
    str(PRED_PATH),
    "--pack", str(DATA_DIR),
    "--split", "dev",
    "--partial"
]

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True,
    check=True
)

print(result.stdout)

##  Close Book System‍‍

In [ ]:
import re
import torch

CLOSED_BOOK_SYSTEM_PROMPT = a1_prompt(CLOSED_CFG)


def build_closed_book_messages(question):
    return [
        {
            "role": "system",
            "content": CLOSED_BOOK_SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": question["question"]
        }
    ]


@torch.inference_mode()
def generate_closed_book_answer(question, max_new_tokens=None):
    if max_new_tokens is None:
        max_new_tokens = CLOSED_CFG["settings"]["generation"]["max_new_tokens"]

    messages = build_closed_book_messages(question)

    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=CLOSED_CFG["settings"]["generation"]["enable_thinking"],
        return_dict=True,
        return_tensors="pt"
    )

    input_length = inputs["input_ids"].shape[1]

    inputs = inputs.to(CLOSED_CFG["settings"]["generation"]["device"])

    outputs = llm.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=CLOSED_CFG["settings"]["generation"]["do_sample"],
        pad_token_id=processor.tokenizer.eos_token_id
    )

    generated_ids = outputs[0][input_length:]

    raw_answer = processor.tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    answer = re.sub(
        r"<think>.*?</think>",
        "",
        raw_answer,
        flags=re.DOTALL
    ).strip()

    answer = re.sub(
        r"^(final answer|answer)\s*:\s*",
        "",
        answer,
        flags=re.IGNORECASE
    ).strip()

    lines = [
        line.strip()
        for line in answer.splitlines()
        if line.strip()
    ]

    return lines[0].strip('"') if lines else ""


In [ ]:

import json
import time
from pathlib import Path

RUN_DIR = a1_prediction_path(CLOSED_CFG, "dev").parent
RUN_DIR.mkdir(parents=True, exist_ok=True)

PRED_PATH = RUN_DIR / "predictions.jsonl"
TIME_PATH = RUN_DIR / "timings.jsonl"

def load_existing(path):
    if not path.exists():
        return []

    with open(path, encoding="utf-8") as f:
        return [
            json.loads(line)
            for line in f
            if line.strip()
        ]

previous = load_existing(PRED_PATH)
completed = {row["qid"] for row in previous}

assert len(previous) == len(completed), "Duplicate qids found"

expected_qids = {q["qid"] for q in questions}
assert completed.issubset(expected_qids)

print("Already completed:", len(completed))

for index, q in enumerate(questions, start=1):

    qid = q["qid"]

    if qid in completed:
        continue

    start = time.perf_counter()

    answer = generate_closed_book_answer(q)

    elapsed = time.perf_counter() - start

    prediction = {
        "qid": qid,
        "answer": answer,
        "retrieved": []
    }

    with open(PRED_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(
            prediction, ensure_ascii=False
        ) + "\n")

    with open(TIME_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps({
            "qid": qid,
            "elapsed_seconds": round(elapsed, 3)
        }) + "\n")

    completed.add(qid)

    print(
        f"[{index}/300] {qid} | "
        f"Answer: {answer} | "
        f"{elapsed:.2f}s"
    )

print("\nCompleted:", len(completed), "/ 300")
print("Saved to:", PRED_PATH)


In [ ]:

import subprocess
import sys
from pathlib import Path

assert len(load_existing(PRED_PATH)) == 300

score_path = next(
    Path("/kaggle/input").rglob("score.py")
)

result = subprocess.run(
    [
        sys.executable,
        str(score_path),
        str(PRED_PATH),
        "--pack", str(DATA_DIR),
        "--split", "dev"
    ],
    capture_output=True,
    text=True,
    check=True
)

print(result.stdout)

In [ ]:

import json
import time
from pathlib import Path

hidden_questions = load_jsonl(
    DATA_DIR / "questions_hidden.jsonl"
)

HIDDEN_DIR = a1_prediction_path(CLOSED_CFG, "hidden").parent
HIDDEN_DIR.mkdir(parents=True, exist_ok=True)

HIDDEN_PRED = HIDDEN_DIR / "predictions.jsonl"
HIDDEN_TIME = HIDDEN_DIR / "timings.jsonl"

def load_existing(path):
    if not path.exists():
        return []

    with open(path, encoding="utf-8") as f:
        return [
            json.loads(line)
            for line in f
            if line.strip()
        ]

previous = load_existing(HIDDEN_PRED)
completed = {r["qid"] for r in previous}

assert len(previous) == len(completed), "Duplicate QIDs"
assert completed.issubset(
    {q["qid"] for q in hidden_questions}
)

print("Hidden questions:", len(hidden_questions))
print("Already completed:", len(completed))

for i, q in enumerate(hidden_questions, start=1):

    qid = q["qid"]

    if qid in completed:
        continue

    start = time.perf_counter()

    answer = generate_closed_book_answer(q)

    elapsed = time.perf_counter() - start

    prediction = {
        "qid": qid,
        "answer": answer,
        "retrieved": []
    }

    with open(HIDDEN_PRED, "a", encoding="utf-8") as f:
        f.write(json.dumps(
            prediction,
            ensure_ascii=False
        ) + "\n")

    with open(HIDDEN_TIME, "a", encoding="utf-8") as f:
        f.write(json.dumps({
            "qid": qid,
            "elapsed_seconds": round(elapsed, 3)
        }) + "\n")

    completed.add(qid)

    print(
        f"[{i}/{len(hidden_questions)}] "
        f"{qid} | {answer} | {elapsed:.1f}s"
    )

print("\nCompleted:", len(completed))
print("Saved:", HIDDEN_PRED)


In [ ]:
import subprocess
import sys

assert len(load_existing(HIDDEN_PRED)) == 300

result = subprocess.run(
    [
        sys.executable,
        str(score_path),
        str(HIDDEN_PRED),
        "--pack", str(DATA_DIR),
        "--split", "hidden",
        "--validate-only"
    ],
    capture_output=True,
    text=True,
    check=True
)

print(result.stdout)


# Oracle Context

In [ ]:

import json
import time
import torch
import subprocess
import sys

from pathlib import Path

assert "llm" in globals() and "processor" in globals(), (
    "Qwen model is not loaded. Load Qwen3.5-9B first."
)

print("Model type:", llm.config.model_type)

DATA_DIR = next(
    Path("/kaggle/input").rglob("labels_dev.jsonl")
).parent

def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [
            json.loads(line)
            for line in f
            if line.strip()
        ]

dev_questions = load_jsonl(
    DATA_DIR / "questions_dev.jsonl"
)
dev_labels = load_jsonl(
    DATA_DIR / "labels_dev.jsonl"
)
corpus = load_jsonl(
    DATA_DIR / "corpus.jsonl"
)

gold_label_map = {
    row["qid"]: row for row in dev_labels
}

oracle_doc_map = {
    doc["doc_id"]: doc for doc in corpus
}

assert len(dev_questions) == 300
assert len(dev_labels) == 300

missing_labels = [
    q["qid"] for q in dev_questions
    if q["qid"] not in gold_label_map
]

missing_docs = [
    doc_id
    for row in dev_labels
    for doc_id in row["gold_doc_ids"]
    if doc_id not in oracle_doc_map
]

assert not missing_labels, missing_labels
assert not missing_docs, missing_docs

print("Questions:", len(dev_questions))
print("Documents:", len(corpus))
print("Labels:", len(dev_labels))
print("Dataset ready.")


In [ ]:

ORACLE_SYSTEM_PROMPT = a1_prompt(ORACLE_CFG)


def build_oracle_messages(question):
    qid = question["qid"]

    # Gold document IDs are used only to select context.
    gold_ids = gold_label_map[qid]["gold_doc_ids"]

    contexts = []

    for i, doc_id in enumerate(gold_ids, start=1):
        doc = oracle_doc_map[doc_id]

        contexts.append(
            f"[Passage {i} | {doc_id}]\n"
            f"Title: {doc['title']}\n"
            f"Text: {doc['text']}"
        )

    user_prompt = (
        "Passages:\n\n"
        + "\n\n".join(contexts)
        + "\n\nQuestion: "
        + question["question"]
        + "\n\nFinal answer:"
    )

    return [
        {
            "role": "system",
            "content": ORACLE_SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]


# Inspect one example before running the experiment
sample = dev_questions[0]
sample_messages = build_oracle_messages(sample)

print("Question:", sample["question"])
print(
    "Number of gold documents:",
    len(gold_label_map[sample["qid"]]["gold_doc_ids"])
)
print("\nPrompt preview:\n")
print(sample_messages[1]["content"][:1500])


In [ ]:

MAX_INPUT_TOKENS = ORACLE_CFG["settings"]["generation"]["max_input_tokens"]

def tokenize_oracle_prompt(question):
    messages = build_oracle_messages(question)

    return processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=ORACLE_CFG["settings"]["generation"]["enable_thinking"],
        return_dict=True,
        return_tensors="pt"
    )


token_lengths = []

for i, q in enumerate(dev_questions, start=1):
    inputs = tokenize_oracle_prompt(q)
    n_tokens = inputs["input_ids"].shape[-1]

    token_lengths.append((q["qid"], n_tokens))

    if i % 50 == 0:
        print(f"Checked {i}/300 prompts")

too_long = [
    (qid, n)
    for qid, n in token_lengths
    if n > MAX_INPUT_TOKENS
]

print("Maximum input tokens:", max(n for _, n in token_lengths))
print("Prompts over limit:", len(too_long))

if too_long:
    print("First long prompts:", too_long[:20])
    raise ValueError(
        "Some Oracle prompts exceed 6144 tokens. "
        "Do not start generation until the context limit is reviewed."
    )

print("All prompts fit within the configured limit.")


In [ ]:

import re
import torch

@torch.inference_mode()
def generate_oracle_answer(question, max_new_tokens=None):
    if max_new_tokens is None:
        max_new_tokens = ORACLE_CFG["settings"]["generation"]["max_new_tokens"]

    inputs = tokenize_oracle_prompt(question)
    input_length = inputs["input_ids"].shape[-1]

    if input_length > MAX_INPUT_TOKENS:
        raise ValueError(
            f"{question['qid']} exceeds context limit: "
            f"{input_length}"
        )

    inputs = inputs.to(ORACLE_CFG["settings"]["generation"]["device"])

    outputs = llm.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=ORACLE_CFG["settings"]["generation"]["do_sample"],
        pad_token_id=processor.tokenizer.eos_token_id
    )

    generated_ids = outputs[0][input_length:]

    raw_answer = processor.tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    answer = re.sub(
        r"<think>.*?</think>",
        "",
        raw_answer,
        flags=re.DOTALL
    ).strip()

    answer = re.sub(
        r"^(final answer|answer)\s*:\s*",
        "",
        answer,
        flags=re.IGNORECASE
    ).strip()

    lines = [
        line.strip()
        for line in answer.splitlines()
        if line.strip()
    ]

    return lines[0].strip('"') if lines else ""


In [ ]:

RUN_DIR = a1_prediction_path(ORACLE_CFG, "dev").parent
RUN_DIR.mkdir(parents=True, exist_ok=True)

PRED_PATH = RUN_DIR / "predictions.jsonl"
TIME_PATH = RUN_DIR / "timings.jsonl"


def read_jsonl(path):
    if not path.exists():
        return []

    with open(path, encoding="utf-8") as f:
        return [
            json.loads(line)
            for line in f
            if line.strip()
        ]


previous = read_jsonl(PRED_PATH)
completed = {row["qid"] for row in previous}

expected_qids = {q["qid"] for q in dev_questions}

assert len(previous) == len(completed), (
    "Duplicate QIDs in saved predictions."
)

assert completed.issubset(expected_qids), (
    "Saved predictions contain unexpected QIDs."
)

print("Already completed:", len(completed))
print("Remaining:", len(expected_qids - completed))


for index, q in enumerate(dev_questions, start=1):

    qid = q["qid"]

    if qid in completed:
        continue

    start = time.perf_counter()

    answer = generate_oracle_answer(q)

    elapsed = time.perf_counter() - start

    prediction = {
        "qid": qid,
        "answer": answer,
        "retrieved": gold_label_map[qid]["gold_doc_ids"]
    }

    timing = {
        "qid": qid,
        "elapsed_seconds": round(elapsed, 3)
    }

    # Save each completed answer immediately
    with open(PRED_PATH, "a", encoding="utf-8") as f:
        f.write(
            json.dumps(prediction, ensure_ascii=False) + "\n"
        )

    with open(TIME_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(timing) + "\n")

    completed.add(qid)

    print(
        f"[{index}/300] {qid} | "
        f"{elapsed:.1f}s | {answer[:100]}"
    )


saved = read_jsonl(PRED_PATH)
saved_qids = {row["qid"] for row in saved}

assert len(saved) == 300
assert saved_qids == expected_qids

print("\nOracle completed successfully.")
print("Predictions:", PRED_PATH)
print("Total:", len(saved))


In [ ]:
score_path = next(
    Path("/kaggle/input").rglob("score.py")
)

assert len(read_jsonl(PRED_PATH)) == 300

result = subprocess.run(
    [
        sys.executable,
        str(score_path),
        str(PRED_PATH),
        "--pack", str(DATA_DIR),
        "--split", "dev"
    ],
    capture_output=True,
    text=True,
    check=True
)

print(result.stdout)

# Dense Retrieval

In [ ]:

import sys
import json
import gc
import hashlib
import subprocess
import importlib.util

from pathlib import Path

import numpy as np
import torch

# Install ChromaDB only if it is not already installed
if importlib.util.find_spec("chromadb") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "chromadb"
    ])

import chromadb

# Locate the course pack
preferred_pack = Path(
    "/kaggle/input/datasets/aminheari/anlp-course-pack/course_pack_v1"
)

if preferred_pack.is_dir():
    DATA_DIR = preferred_pack
else:
    candidates = [
        p.parent
        for p in Path("/kaggle/input").rglob("corpus.jsonl")
        if (p.parent / "questions_dev.jsonl").exists()
        and (p.parent / "labels_dev.jsonl").exists()
    ]

    if len(candidates) != 1:
        raise RuntimeError(
            f"Could not uniquely identify course pack: {candidates}"
        )

    DATA_DIR = candidates[0]


def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [
            json.loads(line)
            for line in f
            if line.strip()
        ]


corpus = load_jsonl(DATA_DIR / "corpus.jsonl")
questions_dev = load_jsonl(DATA_DIR / "questions_dev.jsonl")

doc_ids_original = [str(d["doc_id"]) for d in corpus]

assert len(corpus) == 3882, f"Unexpected corpus size: {len(corpus)}"
assert len(set(doc_ids_original)) == len(doc_ids_original)
assert len(questions_dev) == 300

# Include title and passage text in the document representation
document_texts = [
    f"Title: {d['title']}\nText: {d['text']}"
    for d in corpus
]

# Fingerprint the corpus so stale indexes are not reused
hasher = hashlib.sha256()

for d in corpus:
    for field in ("doc_id", "title", "text"):
        hasher.update(str(d[field]).encode("utf-8"))
        hasher.update(b"\0")

CORPUS_SHA256 = hasher.hexdigest()

print("Course pack:", DATA_DIR)
print("Corpus passages:", len(corpus))
print("Dev questions:", len(questions_dev))
print("Corpus fingerprint:", CORPUS_SHA256[:16])
print("ChromaDB version:", chromadb.__version__)


In [ ]:

import gc
import torch

# Release the LLM from GPU memory; its saved predictions remain intact
if "llm" in globals():
    del llm

# Release previous SentenceTransformer instances, if any
if "model" in globals() and type(model).__name__ == "SentenceTransformer":
    del model

if "embedding_model" in globals():
    del embedding_model

gc.collect()
torch.cuda.empty_cache()

assert torch.cuda.is_available(), (
    "Enable GPU in Kaggle: Settings > Accelerator > GPU"
)

free_bytes, total_bytes = torch.cuda.mem_get_info()

print(f"GPU memory total: {total_bytes / 1024**3:.2f} GB")
print(f"GPU memory free:  {free_bytes / 1024**3:.2f} GB")


In [ ]:
from sentence_transformers import SentenceTransformer

EMBED_MODEL_ID = DENSE_CFG["models"]["embedder"]

embedding_model = SentenceTransformer(
    EMBED_MODEL_ID,
    model_kwargs={"torch_dtype": getattr(torch, DENSE_CFG["settings"]["embedding"]["torch_dtype"])},
    device=DENSE_CFG["settings"]["embedding"]["device"]
)

EMBED_DIM = embedding_model.get_sentence_embedding_dimension()

print("Embedding model:", EMBED_MODEL_ID)
print("Embedding dimension:", EMBED_DIM)
print("Max sequence length:", embedding_model.max_seq_length)


In [ ]:
CHROMA_DIR = Path(
    DENSE_CFG["settings"]["retrieval"]["chroma_dir"]
)

COLLECTION_NAME = DENSE_CFG["settings"]["retrieval"]["collection_name"]

client = chromadb.PersistentClient(
    path=str(CHROMA_DIR)
)

COLLECTION_METADATA = {
    "hnsw:space": DENSE_CFG["settings"]["retrieval"]["space"],
    "embedding_model": EMBED_MODEL_ID,
    "corpus_sha256": CORPUS_SHA256,
    "embedding_dim": int(EMBED_DIM),
    "l2_normalized": True
}


def collection_is_valid(col):
    metadata = col.metadata or {}

    return (
        all(
            metadata.get(key) == value
            for key, value in COLLECTION_METADATA.items()
        )
        and col.count() == len(corpus)
    )


collection = None

try:
    existing = client.get_collection(
        name=COLLECTION_NAME,
        embedding_function=None
    )

    if collection_is_valid(existing):
        collection = existing
        print("Reusing existing persistent collection.")
    else:
        print("Existing collection is stale; rebuilding.")
        client.delete_collection(name=COLLECTION_NAME)

except Exception:
    print("No reusable collection found.")

if collection is None:
    collection = client.create_collection(
        name=COLLECTION_NAME,
        metadata=COLLECTION_METADATA,
        embedding_function=None
    )

    # Encode passages only when building a new index
    document_embeddings = embedding_model.encode(
        document_texts,
        batch_size=DENSE_CFG["settings"]["embedding"]["document_batch_size"],
        precision=DENSE_CFG["settings"]["embedding"]["precision"],
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=DENSE_CFG["settings"]["retrieval"]["normalize_embeddings"]
    )

    document_embeddings

    # Store embeddings, original IDs, passage texts, and titles
    CHROMA_BATCH_SIZE = DENSE_CFG["settings"]["retrieval"]["chroma_batch_size"]

    for start in range(0, len(corpus), CHROMA_BATCH_SIZE):
        end = min(start + CHROMA_BATCH_SIZE, len(corpus))

        batch_docs = corpus[start:end]

        collection.upsert(
            ids=doc_ids_original[start:end],
            embeddings=document_embeddings[start:end].tolist(),
            documents=document_texts[start:end],
            metadatas=[
                {
                    "title": str(d["title"]),
                    "doc_id": str(d["doc_id"])
                }
                for d in batch_docs
            ]
        )

        print(
            f"Stored {end}/{len(corpus)} passages",
            end="\r"
        )

    del document_embeddings
    gc.collect()

assert collection.count() == len(corpus)

print("\nCollection ready.")
print("Stored passages:", collection.count())
print("Persistent directory:", CHROMA_DIR)


In [ ]:
QUERY_INSTRUCTION = DENSE_CFG["settings"]["embedding"]["query_instruction"]

def embed_query(query:str):
    
    formatted_query = (
        f"Instruct: {QUERY_INSTRUCTION}\n"
        f"Query:{query}"
    )
    
    embed_query = embedding_model.encode(
     [formatted_query],
     batch_size=DENSE_CFG["settings"]["embedding"]["query_batch_size"],
     output_value="sentence_embedding",
     convert_to_numpy=True,
     precision=DENSE_CFG["settings"]["embedding"]["precision"],
     normalize_embeddings=DENSE_CFG["settings"]["retrieval"]["normalize_embeddings"]
    )[0]

    return embed_query
    

In [ ]:
from src.retrieval import exact_cosine_search


def dense_search(query, k=None, exclude_ids=frozenset()) -> list[str]:
    if not isinstance(query, str):
        raise TypeError("query must be a string")
    if k is None:
        k = DENSE_K
    if not DENSE_CFG["settings"]["retrieval"]["document_mask"] and exclude_ids:
        raise ValueError("Document mask disabled in config; cannot exclude IDs")
    query_embedding = embed_query(query)
    stored = collection.get(include=["embeddings"], limit=collection.count())
    return exact_cosine_search(
        doc_ids=stored["ids"],
        document_embeddings=stored["embeddings"],
        query_embedding=query_embedding,
        k=k,
        exclude_ids=exclude_ids,
    )


In [ ]:
doc_by_id = {
    str(d["doc_id"]): d
    for d in corpus
}

test_query = questions_dev[0]["question"]

ranked_10 = dense_search(test_query, k=DENSE_K)

assert len(ranked_10) == DENSE_K
assert len(set(ranked_10)) == DENSE_K

print("Question:", test_query)
print("\nTop-10 results:\n")

for rank, doc_id in enumerate(ranked_10, start=1):
    print(
        f"{rank:2d}. {doc_id} | "
        f"{doc_by_id[doc_id]['title']}"
    )

# Verify the mask is applied before top-k
masked_10 = dense_search(
    test_query,
    k=DENSE_K,
    exclude_ids={ranked_10[0]}
)

assert len(masked_10) == DENSE_K
assert ranked_10[0] not in masked_10
assert masked_10[:DENSE_K-1] == ranked_10[1:DENSE_K]

print("\nDocument-mask test: PASSED")
print("Returned 10 results after excluding the original top-1.")


In [ ]:
RUN_DIR = a1_retrieval_path("dev").parent

RUN_DIR.mkdir(parents=True, exist_ok=True)

PRED_PATH = RUN_DIR / "predictions.jsonl"

with open(PRED_PATH, "w", encoding="utf-8") as f:
    for i, q in enumerate(questions_dev, start=1):

        retrieved = dense_search(
            q["question"],
            k=DENSE_K
        )

        row = {
            "qid": q["qid"],
            "answer": "",
            "retrieved": retrieved
        }

        f.write(
            json.dumps(row, ensure_ascii=False) + "\n"
        )

        if i % 25 == 0:
            print(f"Processed {i}/300 questions")

print("Saved:", PRED_PATH)
print("Total predictions:", len(questions_dev))

In [ ]:
SCORE_PATH = Path(
    "/kaggle/input/datasets/hesamgh/nlp-codes/score.py"
)

if not SCORE_PATH.exists():
    candidates = list(
        Path("/kaggle/input").rglob("score.py")
    )

    if len(candidates) != 1:
        raise FileNotFoundError(
            f"Could not uniquely identify score.py: {candidates}"
        )

    SCORE_PATH = candidates[0]

result = subprocess.run(
    [
        sys.executable,
        str(SCORE_PATH),
        str(PRED_PATH),
        "--pack", str(DATA_DIR),
        "--split", "dev"
    ],
    capture_output=True,
    text=True,
    check=True
)

print(result.stdout)

# DenseRAG + LLM

In [ ]:

import json
import gc
import re
import torch
from pathlib import Path

RETRIEVAL_PATH = a1_retrieval_path("dev")

with open(RETRIEVAL_PATH, encoding="utf-8") as f:
    retrieval_rows = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

question_by_qid = {
    q["qid"]: q
    for q in questions_dev
}

assert len(retrieval_rows) == 300
assert all(row["qid"] in question_by_qid for row in retrieval_rows)
assert all(len(row["retrieved"]) == DENSE_K for row in retrieval_rows)

print("Questions:", len(retrieval_rows))
print("Retrieved documents per question:", DENSE_K)
print("First question:", question_by_qid[retrieval_rows[0]["qid"]]["question"])
print("Retrieved IDs:", retrieval_rows[0]["retrieved"])


In [ ]:

if "embedding_model" in globals():
    del embedding_model

gc.collect()
torch.cuda.empty_cache()

print("LLM loaded:", "llm" in globals())
print("Processor loaded:", "processor" in globals())
print("GPU available:", torch.cuda.is_available())


In [ ]:

SYSTEM_PROMPT = a1_prompt(DENSE_CFG)


MAX_INPUT_TOKENS = DENSE_CFG["settings"]["generation"]["max_input_tokens"]


def build_rag_messages(question, doc_ids):

    passage_blocks = []

    for rank, doc_id in enumerate(doc_ids, start=1):
        if doc_id not in doc_by_id:
            raise KeyError(f"Unknown document ID: {doc_id}")

        doc = doc_by_id[doc_id]

        passage_blocks.append(
            f"[Passage {rank} | {doc_id}]\n"
            f"Title: {doc['title']}\n"
            f"Text: {doc['text']}"
        )

    passages_text = "\n\n".join(passage_blocks)

    user_prompt = (
        f"Passages:\n\n{passages_text}\n\n"
        f"Question: {question}\n\n"
        "Final answer:"
    )

    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt}
    ]


def tokenize_rag_prompt(question, doc_ids):

    # Preserve the original retrieval ranking.
    # If the context is too long, remove the lowest-ranked
    # passage from the LLM context, without changing retrieved IDs.
    for n in range(min(len(doc_ids), DENSE_CFG["settings"]["generation"]["context_top_k"]), 0, -1):

        messages = build_rag_messages(
            question,
            doc_ids[:n]
        )

        inputs = processor.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            enable_thinking=DENSE_CFG["settings"]["generation"]["enable_thinking"],
            return_dict=True,
            return_tensors="pt"
        )

        input_length = inputs["input_ids"].shape[1]

        if input_length <= MAX_INPUT_TOKENS:
            return inputs, n

    raise ValueError(
        "The prompt exceeds the input-token limit, "
        "even with only the top-ranked passage."
    )


def generate_dense_rag_answer(
    question,
    doc_ids,
    max_new_tokens=None
):

    if max_new_tokens is None:
        max_new_tokens = DENSE_CFG["settings"]["generation"]["max_new_tokens"]

    if "llm" not in globals() or "processor" not in globals():
        raise RuntimeError(
            "Load Qwen3.5-9B and its processor first."
        )

    inputs, n_context = tokenize_rag_prompt(
        question,
        doc_ids
    )

    inputs = {
        key: value.to(DENSE_CFG["settings"]["generation"]["device"])
        for key, value in inputs.items()
    }

    input_length = inputs["input_ids"].shape[1]

    with torch.inference_mode():
        outputs = llm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=DENSE_CFG["settings"]["generation"]["do_sample"],
            pad_token_id=processor.tokenizer.eos_token_id
        )

    generated_ids = outputs[0][input_length:]

    answer_text = processor.tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    # Remove any residual thinking tags or answer prefixes.
    answer_text = re.sub(
        r"<think>.*?</think>",
        "",
        answer_text,
        flags=re.DOTALL
    ).strip()

    answer_text = answer_text.replace("<think>", "").replace(
        "</think>", ""
    ).strip()

    for prefix in ("Final answer:", "Answer:"):
        if answer_text.lower().startswith(prefix.lower()):
            answer_text = answer_text[len(prefix):].strip()

    # The task expects a short answer.
    answer_text = answer_text.splitlines()[0].strip() if answer_text else ""

    return answer_text


In [ ]:

import torch
from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,
    BitsAndBytesConfig
)

MODEL_ID = DENSE_CFG["models"]["llm"]["name"]

assert torch.cuda.is_available(), (
    "GPU is not enabled. Go to Kaggle Settings > Accelerator."
)

# 4-bit quantization
quant_config = BitsAndBytesConfig(
    load_in_4bit=DENSE_CFG["models"]["llm"]["load_in_4bit"],
    bnb_4bit_quant_type=DENSE_CFG["models"]["llm"]["quantization"],
    bnb_4bit_compute_dtype=a1_torch_dtype(torch, DENSE_CFG),
    bnb_4bit_use_double_quant=DENSE_CFG["models"]["llm"]["double_quantization"]
)

print("Loading processor...")

processor = AutoProcessor.from_pretrained(
    MODEL_ID
)

print("Loading Qwen3.5-9B...")

llm = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    dtype=getattr(torch, DENSE_CFG["models"]["llm"]["model_dtype"]),
    device_map=DENSE_CFG["models"]["llm"]["device_map"],
    attn_implementation=DENSE_CFG["models"]["llm"]["attention_implementation"],
    low_cpu_mem_usage=DENSE_CFG["models"]["llm"]["low_cpu_mem_usage"]
)

llm.eval()

print("Model loaded successfully.")
print("Model device:", llm.device)
print("Processor:", type(processor).__name__)


In [ ]:

import json
import subprocess
import sys
from pathlib import Path

# --------------------------------------------------
# 1. Load the 300 Dev questions and Dense retrieval
# --------------------------------------------------

DATA_DIR = Path(
    "/kaggle/input/datasets/aminheari/anlp-course-pack/course_pack_v1"
)

RETRIEVAL_PATH = a1_retrieval_path("dev")

OUT_DIR = a1_prediction_path(DENSE_CFG, "dev").parent

OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_PATH = OUT_DIR / "predictions.jsonl"

assert RETRIEVAL_PATH.exists(), "Dense retrieval file not found."
assert "llm" in globals(), "Load the Qwen LLM first."
assert "processor" in globals(), "Load the processor first."
assert "generate_dense_rag_answer" in globals(), (
    "Define generate_dense_rag_answer first."
)

with open(
    DATA_DIR / "questions_dev.jsonl",
    encoding="utf-8"
) as f:
    questions_dev = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

with open(RETRIEVAL_PATH, encoding="utf-8") as f:
    retrieval_rows = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

assert len(questions_dev) == 300
assert len(retrieval_rows) == 300

question_by_qid = {
    q["qid"]: q
    for q in questions_dev
}

assert len(question_by_qid) == 300

retrieval_by_qid = {
    row["qid"]: row
    for row in retrieval_rows
}

assert len(retrieval_by_qid) == 300
assert set(retrieval_by_qid) == set(question_by_qid)

print("Dev questions:", len(questions_dev))
print("Retrieval records:", len(retrieval_rows))
print("Output:", OUT_PATH)


# --------------------------------------------------
# 2. Resume from already saved predictions
# --------------------------------------------------

done_qids = set()

if OUT_PATH.exists():
    with open(OUT_PATH, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                saved = json.loads(line)
                done_qids.add(saved["qid"])

    unknown_qids = done_qids - set(question_by_qid)

    if unknown_qids:
        raise ValueError(
            f"Output contains unexpected QIDs: {unknown_qids}"
        )

print("Already completed:", len(done_qids))
print("Remaining:", 300 - len(done_qids))


# --------------------------------------------------
# 3. Generate answers using retrieved passages
# --------------------------------------------------

with open(OUT_PATH, "a", encoding="utf-8") as f:

    for i, q in enumerate(questions_dev, start=1):

        qid = q["qid"]

        if qid in done_qids:
            continue

        retrieved = retrieval_by_qid[qid]["retrieved"]

        assert len(retrieved) == DENSE_K
        assert len(set(retrieved)) == DENSE_K

        answer_text = generate_dense_rag_answer(
            question=q["question"],
            doc_ids=retrieved,
            max_new_tokens=DENSE_CFG["settings"]["generation"]["max_new_tokens"]
        )

        row = {
            "qid": qid,
            "answer": answer_text,
            "retrieved": retrieved
        }

        f.write(json.dumps(row, ensure_ascii=False) + "\n")
        f.flush()

        done_qids.add(qid)

        if len(done_qids) % 10 == 0:
            print(
                f"Completed {len(done_qids)}/300 | "
                f"Latest QID: {qid} | "
                f"Answer: {answer_text[:100]!r}"
            )

print("\nGeneration finished.")
print("Saved predictions:", OUT_PATH)
print("Completed questions:", len(done_qids))


In [ ]:

SCORE_PATH = Path(
    "/kaggle/input/datasets/hesamgh/nlp-codes/score.py"
)

if not SCORE_PATH.exists():
    candidates = list(
        Path("/kaggle/input").rglob("score.py")
    )

    if len(candidates) != 1:
        raise FileNotFoundError(
            f"Could not uniquely identify score.py: {candidates}"
        )

    SCORE_PATH = candidates[0]

result = subprocess.run(
    [
        sys.executable,
        str(SCORE_PATH),
        str(OUT_PATH),
        "--pack", str(DATA_DIR),
        "--split", "dev"
    ],
    capture_output=True,
    text=True,
    check=True
)

print(result.stdout)


In [ ]:

import json
from pathlib import Path
from tqdm.auto import tqdm

PACK_DIR = Path(
    "/kaggle/input/datasets/aminheari/anlp-course-pack/course_pack_v1"
)

QUESTIONS_PATH = PACK_DIR / "questions_hidden.jsonl"

RETRIEVAL_DIR = a1_retrieval_path("hidden").parent
RETRIEVAL_PATH = RETRIEVAL_DIR / "predictions.jsonl"

RETRIEVAL_DIR.mkdir(parents=True, exist_ok=True)

assert "embedding_model" in globals(), "Embedding model is not loaded."
assert "dense_search" in globals(), "dense_search is not defined."
assert "doc_by_id" in globals(), "doc_by_id is not defined."

with open(QUESTIONS_PATH, encoding="utf-8") as f:
    questions_hidden = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

assert len(questions_hidden) == 300, (
    f"Expected 300 questions, got {len(questions_hidden)}"
)

qid_set = {q["qid"] for q in questions_hidden}

# Resume previously completed retrievals when possible.
retrieval_by_qid = {}

if RETRIEVAL_PATH.exists():
    with open(RETRIEVAL_PATH, encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue

            row = json.loads(line)
            qid = row.get("qid")
            ids = row.get("retrieved", [])

            if (
                qid in qid_set
                and isinstance(ids, list)
                and len(ids) == DENSE_K
                and len(set(ids)) == DENSE_K
                and all(doc_id in doc_by_id for doc_id in ids)
            ):
                retrieval_by_qid[qid] = ids

with open(RETRIEVAL_PATH, "a", encoding="utf-8") as f:
    for q in tqdm(questions_hidden, desc="Dense retrieval on Hidden"):

        qid = q["qid"]

        if qid in retrieval_by_qid:
            continue

        doc_ids = dense_search(q["question"], k=DENSE_K)

        if len(doc_ids) != DENSE_K:
            raise RuntimeError(
                f"{qid}: expected 10 documents, got {len(doc_ids)}"
            )

        if len(set(doc_ids)) != DENSE_K:
            raise RuntimeError(f"{qid}: duplicate document IDs.")

        if not all(doc_id in doc_by_id for doc_id in doc_ids):
            raise RuntimeError(f"{qid}: unknown document ID.")

        retrieval_by_qid[qid] = doc_ids

        f.write(json.dumps({
            "qid": qid,
            "retrieved": doc_ids
        }, ensure_ascii=False) + "\n")

        f.flush()

# Save exactly 300 retrieval records in the original question order.
with open(RETRIEVAL_PATH, "w", encoding="utf-8") as f:
    for q in questions_hidden:
        json.dump({
            "qid": q["qid"],
            "retrieved": retrieval_by_qid[q["qid"]]
        }, f, ensure_ascii=False)
        f.write("\n")

assert len(retrieval_by_qid) == 300

print("Hidden retrieval completed.")
print("Questions:", len(retrieval_by_qid))
print("Saved to:", RETRIEVAL_PATH)


In [ ]:

import gc
import torch

# Release the embedding model before loading the LLM.
del embedding_model

gc.collect()
torch.cuda.empty_cache()

print("Embedding model released.")
print("GPU memory allocated:", round(
    torch.cuda.memory_allocated() / 1024**3, 2
), "GB")


In [ ]:

from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,
    BitsAndBytesConfig
)

MODEL_ID = DENSE_CFG["models"]["llm"]["name"]

if "processor" not in globals():
    processor = AutoProcessor.from_pretrained(MODEL_ID)

quant_config = BitsAndBytesConfig(
    load_in_4bit=DENSE_CFG["models"]["llm"]["load_in_4bit"],
    bnb_4bit_quant_type=DENSE_CFG["models"]["llm"]["quantization"],
    bnb_4bit_compute_dtype=a1_torch_dtype(torch, DENSE_CFG),
    bnb_4bit_use_double_quant=DENSE_CFG["models"]["llm"]["double_quantization"]
)

llm = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    dtype=getattr(torch, DENSE_CFG["models"]["llm"]["model_dtype"]),
    device_map=DENSE_CFG["models"]["llm"]["device_map"],
    attn_implementation=DENSE_CFG["models"]["llm"]["attention_implementation"],
    low_cpu_mem_usage=DENSE_CFG["models"]["llm"]["low_cpu_mem_usage"]
)

llm.eval()

print("LLM loaded successfully.")
print("Device:", llm.device)


In [ ]:

OUT_DIR = a1_prediction_path(DENSE_CFG, "hidden").parent
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_PATH = OUT_DIR / "predictions.jsonl"

# Read the saved retrieval results.
with open(RETRIEVAL_PATH, encoding="utf-8") as f:
    retrieval_rows = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

retrieval_by_qid = {
    row["qid"]: row["retrieved"]
    for row in retrieval_rows
}

assert len(retrieval_by_qid) == 300

# Resume completed answers if available and compatible.
answer_by_qid = {}

if OUT_PATH.exists():
    with open(OUT_PATH, encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue

            row = json.loads(line)
            qid = row.get("qid")

            if (
                qid in retrieval_by_qid
                and row.get("retrieved") == retrieval_by_qid[qid]
                and isinstance(row.get("answer"), str)
            ):
                answer_by_qid[qid] = row

with open(OUT_PATH, "a", encoding="utf-8") as f:
    for q in tqdm(questions_hidden, desc="Generating Hidden answers"):

        qid = q["qid"]

        if qid in answer_by_qid:
            continue

        answer = generate_dense_rag_answer(
            question=q["question"],
            doc_ids=retrieval_by_qid[qid],
            max_new_tokens=DENSE_CFG["settings"]["generation"]["max_new_tokens"]
        )

        if isinstance(answer, dict):
            answer = answer.get("answer", "")

        row = {
            "qid": qid,
            "answer": str(answer).strip(),
            "retrieved": retrieval_by_qid[qid]
        }

        answer_by_qid[qid] = row

        f.write(json.dumps(row, ensure_ascii=False) + "\n")
        f.flush()

# Write a clean final file in the original question order.
with open(OUT_PATH, "w", encoding="utf-8") as f:
    for q in questions_hidden:
        row = answer_by_qid[q["qid"]]
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

assert len(answer_by_qid) == 300

print("Hidden Dense RAG completed.")
print("Predictions:", len(answer_by_qid))
print("Output:", OUT_PATH)


In [ ]:

import subprocess
import sys
from pathlib import Path

SCORE_PATH = Path(
    "/kaggle/input/datasets/hesamgh/nlp-codes/score.py"
)

PRED_PATH = Path(
    "/kaggle/working/runs/hidden/dense_rag_qwen3_5_9b/predictions.jsonl"
)

PACK_DIR = Path(
    "/kaggle/input/datasets/aminheari/anlp-course-pack/course_pack_v1"
)

# Verify that all required files exist.
assert SCORE_PATH.exists(), f"score.py not found: {SCORE_PATH}"
assert PRED_PATH.exists(), f"Predictions not found: {PRED_PATH}"
assert PACK_DIR.exists(), f"Course pack not found: {PACK_DIR}"

cmd = [
    sys.executable,
    str(SCORE_PATH),
    str(PRED_PATH),
    "--pack", str(PACK_DIR),
    "--split", "hidden",
    "--validate-only"
]

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True
)

print(result.stdout)

if result.stderr:
    print("STDERR:")
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        f"Validation failed with exit code {result.returncode}"
    )

print("Hidden predictions validation completed.")


In [ ]:

from pathlib import Path
import zipfile

base = Path("/kaggle/working")
zip_path = base / "project_backup.zip"

files = [
    p for p in base.rglob("*")
    if p.is_file() and p != zip_path
]

with zipfile.ZipFile(
    zip_path, "w", zipfile.ZIP_DEFLATED
) as z:
    for p in files:
        z.write(p, p.relative_to(base))

print("ZIP created successfully")
print("Path:", zip_path)
print("Files included:", len(files))
print("Size (MB):", round(zip_path.stat().st_size / 1024**2, 2))
